# Colab Training Notebook for spiking-transformer-cs4782
# This notebook sets up a Colab GPU runtime, prepares enwik8, and runs training.


In [1]:
from google.colab import drive
drive.mount('/content/drive')

!cp -r /content/drive/MyDrive/DeepLearning/spikegpt/spiking-transformer-cs4782 /content/
%cd /content/spiking-transformer-cs4782

Mounted at /content/drive
/content/spiking-transformer-cs4782


In [2]:

!pip install -r requirements.txt
!pip install spikingjelly


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 437.6/437.6 kB 8.7 MB/s eta 0:00:00


In [3]:
import torch
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))


torch: 2.10.0+cu128
cuda available: True
gpu: Tesla T4


In [4]:
!wget http://mattmahoney.net/dc/enwik8.zip
!unzip -o enwik8.zip


--2026-04-26 20:52:30--  http://mattmahoney.net/dc/enwik8.zip
Resolving mattmahoney.net (mattmahoney.net)... 20.119.76.151
Connecting to mattmahoney.net (mattmahoney.net)|20.119.76.151|:80... connected.
HTTP request sent, awaiting response... 200 OK
Length: 36445475 (35M) [application/zip]
Saving to: ‘enwik8.zip’

enwik8.zip          100%[===================>]  34.76M  50.9MB/s    in 0.7s    

2026-04-26 20:52:30 (50.9 MB/s) - ‘enwik8.zip’ saved [36445475/36445475]

Archive:  enwik8.zip
  inflating: enwik8                  


In [5]:
from pathlib import Path

raw = Path("enwik8").read_bytes()

out_dir = Path("data/enwik8_split")
out_dir.mkdir(parents=True, exist_ok=True)

n = len(raw)
train_end = int(0.9 * n)
valid_end = int(0.95 * n)

(out_dir / "train.txt").write_bytes(raw[:train_end])
(out_dir / "valid.txt").write_bytes(raw[train_end:valid_end])
(out_dir / "test.txt").write_bytes(raw[valid_end:])

print("done")
print("train:", (out_dir / "train.txt").stat().st_size)
print("valid:", (out_dir / "valid.txt").stat().st_size)
print("test:", (out_dir / "test.txt").stat().st_size)


done
train: 90000000
valid: 5000000
test: 5000000


In [6]:
# cell to update all the code if you change it
import sys
import importlib
from pathlib import Path

code_path = str(Path("code").resolve())
if code_path not in sys.path:
    sys.path.insert(0, code_path)

for name in ["model", "config", "dataset", "train", "test", "generate"]:
    if name in sys.modules:
        importlib.reload(sys.modules[name])

import model
import config
import dataset
import train
import test
import generate

from model import SpikingGPT
from config import GPTConfig, TrainerConfig
from dataset import Enwik8Dataset

print("project modules refreshed")



project modules refreshed


In [7]:
# test before actual training
import sys
from pathlib import Path
import torch

sys.path.insert(0, str(Path("code").resolve()))

from config import GPTConfig
from model import SpikingGPT
from dataset import Enwik8Dataset

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

cfg = GPTConfig(ctx_len=64, n_embd=64, n_layer=2)

dataset = Enwik8Dataset("data/enwik8_split/train.txt", cfg.ctx_len)
x, y = dataset[0]
x = x.unsqueeze(0).to(device)
y = y.unsqueeze(0).to(device)

model = SpikingGPT(cfg).to(device)

print("x shape:", x.shape)
print("y shape:", y.shape)

loss = model(x, y)
print("smoke test loss:", loss.item())


device: cuda
x shape: torch.Size([1, 64])
y shape: torch.Size([1, 64])
smoke test loss: 5.652362823486328


In [8]:
!ls data/enwik8_split


test.txt  train.txt  valid.txt


In [ ]:
#Resume from latest completed epoch:
!python3 code/train.py --resume latest --checkpoint-dir /content/drive/MyDrive/spiking-transformer-cs4782-results/checkpoints


In [ ]:
#Start from scratch:
!python3 code/train.py --resume none --checkpoint-dir /content/drive/MyDrive/spiking-transformer-cs4782-results/checkpoints


In [ ]:
#Resume from a specific completed checkpoint:
!python3 code/train.py --resume-path /content/drive/MyDrive/spiking-transformer-cs4782-results/checkpoints/epoch_4.pt --checkpoint-dir /content/drive/MyDrive/spiking-transformer-cs4782-results/checkpoints


In [ ]:
!mkdir -p /content/drive/MyDrive/spiking-transformer-cs4782-results
!cp -r results /content/drive/MyDrive/spiking-transformer-cs4782-results/


In [ ]:
!python code/test.py


available checkpoints:
 - epoch_1.pt
using checkpoint: epoch_1.pt
test_loss=2.1212
test_perplexity=8.3413
test_bpc=3.0603
R_hat=0.187486
params_m=3.5512

Energy Table Inputs
T=256
d=256
R_hat=0.187486
E_MAC=4.5 pJ
E_AC=0.9 pJ

Energy Table Values
Q/R,K,V: vanilla = E_MAC * 3*T*d^2 = 4.5 * 3 * 256 * 256^2 = 2.2649e+08 pJ
Q/R,K,V: spikegpt = E_AC * R_hat * 3*T*d^2 = 0.9 * 0.187486 * 3 * 256 * 256^2 = 8.4928e+06 pJ
f(Q/R,K,V): vanilla = E_MAC * 2*T^2*d = 4.5 * 2 * 256^2 * 256 = 1.5099e+08 pJ
f(Q/R,K,V): spikegpt = E_MAC * 7*T*d = 4.5 * 7 * 256 * 256 = 2.0644e+06 pJ
Scale: vanilla = E_MAC * T^2 = 4.5 * 256^2 = 2.9491e+05 pJ
Softmax: vanilla = E_MAC * 2*T^2 = 4.5 * 2 * 256^2 = 5.8982e+05 pJ
FFN layer 1: vanilla = E_MAC * FL_MLP1 = 4.5 * (256 * 1024) = 1.1796e+06 pJ
FFN layer 1: spikegpt = E_AC * R_hat * FL_MLP1 = 0.9 * 0.187486 * (256 * 1024) = 4.4233e+04 pJ
FFN layer 2: vanilla = E_MAC * FL_MLP2 = 4.5 * (256 * 256) = 2.9491e+05 pJ
FFN layer 2: spikegpt = E_AC * R_hat * FL_MLP2 = 0.9 * 0.18

In [ ]:
!python code/generate_colab.py

using device: cuda
using checkpoint: epoch_1.pt
ctx_len=256 n_embd=256 n_layer=4

prompt:
'The '

generated:
The same and such and new it to the gave and which a muring of the orden and the the from the creach such to most and same to shous are of the constand simple is a relevision of sechnatic was negal are th
